### Operational Dashboard App

Deploys the Casper's Operational Dashboard Databricks App. Grants SQL warehouse access, wires the
Lakebase database resource, and deploys the FastAPI app from apps/caspers-ops-dashboard/.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
# Shared Lakebase Autoscaling project provisioned by stages/lakebase_project.ipynb.
# The Operational Dashboard app reads/writes the `caspers_ops` database
# within this project; see stages/operational_lakebase.ipynb for the
# sessions/messages table DDL.
OPS_PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
OPS_ENDPOINT_PATH = f"projects/{OPS_PROJECT_ID}/branches/production/endpoints/primary"
# Lakebase Autoscale requires DNS-safe names (no underscores).
OPS_DATABASE_NAME = "caspers-ops"

In [ ]:
import sys, os
sys.path.append('../utils')
from uc_state import add

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import (
    App, AppResource, AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    AppDeployment,
)
from databricks.sdk.service import catalog as catalog_svc

w = WorkspaceClient()

# Find the ops warehouse.  DABs owns this resource — it's defined under
# `resources.sql_warehouses.caspers_ops_warehouse` in databricks.yml and
# created by `bundle deploy -t all`.  This stage is now find-only: if the
# warehouse is missing, deploy hasn't been run (or the resource was removed).
# Cleanup is also DABs-owned (`bundle destroy -t all`), so we no longer
# register the warehouse with uc_state to avoid double-delete.
WAREHOUSE_NAME = f"{CATALOG}-ops-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if not existing_wh:
    raise RuntimeError(
        f"Warehouse '{WAREHOUSE_NAME}' not found. It is created by "
        f"`databricks bundle deploy -t all --var catalog={CATALOG}` as the "
        f"`caspers_ops_warehouse` DABs resource. Deploy the bundle first."
    )
warehouse = existing_wh[0]
print(f"\u267b\ufe0f Using DABs-managed warehouse: {warehouse.id} ({WAREHOUSE_NAME})")

In [ ]:
source_code_path = os.path.abspath("../apps/caspers-ops-dashboard")

# P1-18: catalog-scope the app name so two users on the same workspace don't
# fight over the global `caspers-ops-dashboard` slug.  Use the shorter
# `ops-dashboard-` prefix to keep the full name under the 30-char Databricks
# Apps limit even for long catalog names.
import re as _re
APP_NAME = _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9-]", "-", f"ops-dashboard-{CATALOG}".lower())).strip("-")[:30]
print(f"App name: {APP_NAME}")

# Lakebase Autoscale does not support AppResourceDatabase — the app connects
# directly using w.postgres.generate_database_credential() with the endpoint path.
app_def = App(
    name=APP_NAME,
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
            ),
        ),
    ],
)

import time

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"\u267b\ufe0f App {APP_NAME} exists, updating...")
    w.apps.update(APP_NAME, app_def)
except Exception:
    w.apps.create(app_def)

def _app_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    if s is None:
        s = getattr(a, "state", None)
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current = w.apps.get(APP_NAME)
    state = _app_state(current)
    print(f"App {APP_NAME} state: {state}")
    if state in ("ACTIVE", "RUNNING", "READY"):
        app_status = current
        break
    if state in ("ERROR", "FAILED"):
        raise RuntimeError(f"App {APP_NAME} entered failure state: {state}")
    if time.time() > deadline:
        raise TimeoutError(f"App {APP_NAME} not ready after 30 minutes (last state: {state})")
    time.sleep(15)

add(CATALOG, "apps", app_status)
print(f"\u2705 App {APP_NAME} ready")

# Resolve SP ID immediately — used in all subsequent permission cells.
# service_principal_client_id is the documented OAuth UUID field.
app_sp_id = (
    getattr(app_status, 'service_principal_client_id', None)
    or (app_status.as_dict() if hasattr(app_status, 'as_dict') else {}).get('service_principal_client_id')
    or (app_status.as_dict() if hasattr(app_status, 'as_dict') else {}).get('id')
)
assert app_sp_id, "Could not determine app service principal ID"
print(f"\U0001f511 App SP ID: {app_sp_id}")

In [ ]:
# Grant app permissions to relevant UC resources
for full_name, securable_type, privilege in [
    (CATALOG, "CATALOG", catalog_svc.Privilege.USE_CATALOG),
    (f"{CATALOG}.lakeflow", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    # all_events raw table
    (f"{CATALOG}.lakeflow.all_events", "TABLE", catalog_svc.Privilege.SELECT),
    # Gold/silver tables queried by Genie revenue & ops spaces
    (f"{CATALOG}.lakeflow.gold_brand_sales_day", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.gold_location_sales_hourly", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.gold_order_header", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.silver_order_items", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    # Simulator dimension tables queried by Genie spaces
    (f"{CATALOG}.simulator.brand_locations", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.brands", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.items", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.locations", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.food_safety", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.food_safety.inspections", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.food_safety.violations", "TABLE", catalog_svc.Privilege.SELECT),
    # Document schemas — needed so the app can list PDFs from UC volumes
    (f"{CATALOG}.legal_complaints", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.regulatory", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.audits", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.consultancy", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.legal_complaints", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.regulatory", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.audits", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.consultancy", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
]:
    try:
        w.grants.update(
            full_name=full_name,
            securable_type=securable_type,
            changes=[
                catalog_svc.PermissionsChange(
                    add=[privilege],
                    principal=app_sp_id,
                )
            ],
        )
        print(f"\u2705 Granted {privilege} on {full_name}")
    except Exception as e:
        print(f"\u26a0\ufe0f Could not grant {privilege} on {full_name}: {e}")

In [ ]:
import json, time
from databricks.sdk.service.serving import (
    ServingEndpointAccessControlRequest,
    ServingEndpointPermissionLevel,
)

def _grant_can_query(endpoint_id: str, endpoint_name: str, sp_id: str, retries: int = 3) -> bool:
    """Grant CAN_QUERY on a serving endpoint.
    endpoint_id  — the UUID returned by the list API (required by update_permissions)
    endpoint_name — human-readable name used only for logging
    """
    for attempt in range(retries):
        try:
            w.serving_endpoints.update_permissions(
                serving_endpoint_id=endpoint_id,   # must be UUID, not name
                access_control_list=[
                    ServingEndpointAccessControlRequest(
                        service_principal_name=sp_id,
                        permission_level=ServingEndpointPermissionLevel.CAN_QUERY,
                    )
                ],
            )
            print(f"\u2705 Granted CAN_QUERY on {endpoint_name} ({endpoint_id})")
            return True
        except Exception as e:
            if attempt < retries - 1:
                print(f"  Retry {attempt+1}/{retries} for {endpoint_name}: {e}")
                time.sleep(5 * (attempt + 1))
            else:
                print(f"\u274c FAILED to grant CAN_QUERY on {endpoint_name}: {e}")
                return False

# Markers in error messages that indicate the Genie space no longer exists
# or the current identity has no permission to grant on it. These are almost
# always stale uc_state rows from previous deploys (the space was deleted, or
# created under a different identity that no longer owns it). We warn + skip
# rather than failing the whole stage.
_GENIE_ORPHAN_MARKERS = (
    "RESOURCE_DOES_NOT_EXIST",
    "NOT_FOUND",
    " 404",
    "PERMISSION_DENIED",
    " 403",
    "does not exist",
    "cannot be found",
)

def _grant_can_run_genie(space_id: str, sp_id: str, retries: int = 3) -> str:
    """Grant CAN_RUN on a Genie space.
    Returns one of: 'ok', 'orphan' (skip), 'failed' (real failure).
    """
    for attempt in range(retries):
        try:
            w.api_client.do(
                "PATCH",
                f"/api/2.0/permissions/genie/{space_id}",
                body={"access_control_list": [
                    {"service_principal_name": sp_id, "permission_level": "CAN_RUN"}
                ]},
            )
            print(f"\u2705 Granted CAN_RUN on Genie space {space_id} to {sp_id}")
            return "ok"
        except Exception as e:
            msg = str(e)
            if any(marker in msg for marker in _GENIE_ORPHAN_MARKERS):
                first_line = msg.splitlines()[0] if msg else ""
                print(f"\u26a0\ufe0f  Skipping orphan Genie space {space_id} (stale uc_state row): {first_line}")
                return "orphan"
            if attempt < retries - 1:
                print(f"  Retry {attempt+1}/{retries} for Genie space {space_id}: {e}")
                time.sleep(5 * (attempt + 1))
            else:
                print(f"\u274c FAILED to grant CAN_RUN on Genie space {space_id}: {e}")
                return "failed"

# ── Resolve endpoints + Genie spaces created BY THIS CATALOG from uc_state ────
# P1-16: previously this cell scanned every mas-*/ka-* endpoint and every Genie
# space in the workspace and granted to the app SP.  Two pain points:
#   1. Cross-tenant blast radius — a colleague's `mas-xxxxxxxx-endpoint` in the
#      same workspace would silently get our app SP granted CAN_QUERY.
#   2. The grant list grew unboundedly with every catalog deployed, so
#      runtime got slower and quota errors started surfacing.
# uc_state stores the resources the current catalog's stages created — scope
# the grants to that set only.
import json as _json

def _uc_state_rows(resource_type):
    try:
        rows = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = '{resource_type}'
            ORDER BY created_at DESC
        """).collect()
        return [_json.loads(r.resource_data) for r in rows]
    except Exception as e:
        print(f"⚠️  uc_state lookup failed for {resource_type}: {e}")
        return []

# Build the set of endpoint names this catalog owns.
# - multi_agent_supervisors: row has `endpoint_name`
# - knowledge_assistants:    row has `tile_id` → endpoint name is `ka-{tile_id[:8]}-endpoint`
owned_endpoint_names = set()
for row in _uc_state_rows("multi_agent_supervisors"):
    n = row.get("endpoint_name")
    if n:
        owned_endpoint_names.add(n)
for row in _uc_state_rows("knowledge_assistants"):
    tile_id = row.get("tile_id") or ""
    if tile_id:
        owned_endpoint_names.add(f"ka-{tile_id[:8]}-endpoint")

# Build the set of Genie spaces this catalog owns (`space_id`).
# uc_state may contain stale rows from prior deploys: the `genie_spaces` stage
# is non-idempotent and creates a fresh trio of spaces on every run, so the
# table can accumulate many distinct space_ids that share the same title.
# Granting on all of them triggers RuntimeError on the orphans. Dedupe by
# title and keep only the most-recent space_id per title (rows come from
# `_uc_state_rows` already ordered by created_at DESC).
owned_space_ids = set()
_seen_titles = set()
_stale_space_count = 0
for row in _uc_state_rows("genie_spaces"):
    sid = row.get("space_id")
    if not sid:
        continue
    title = row.get("title") or sid  # fall back to id if title missing
    if title in _seen_titles:
        _stale_space_count += 1
        continue
    _seen_titles.add(title)
    owned_space_ids.add(sid)
if _stale_space_count:
    print(
        f"\u26a0\ufe0f  uc_state has {_stale_space_count} stale genie_spaces rows "
        f"(older space_ids for the same titles); ignoring and granting only on "
        f"the {len(owned_space_ids)} most-recent space(s) per title."
    )

print(f"uc_state: {len(owned_endpoint_names)} owned endpoints, {len(owned_space_ids)} owned Genie spaces")

# ── Grant CAN_QUERY on owned mas-*/ka-* serving endpoints ─────────────────────
# IMPORTANT: update_permissions requires the endpoint UUID (ep.id), not the name.
# We still call w.serving_endpoints.list() because the permissions API needs
# the UUID, but we filter to only the names we own.
ep_failures = []
all_eps = list(w.serving_endpoints.list())
target_eps = [ep for ep in all_eps if (ep.name or "") in owned_endpoint_names]
# Warn if uc_state references an endpoint that doesn't exist in the workspace
# (catches drift between uc_state and reality, e.g. someone deleted an endpoint).
missing = owned_endpoint_names - {ep.name for ep in target_eps}
if missing:
    print(f"⚠️  uc_state lists endpoints not present in workspace (skipping): {sorted(missing)}")
print(f"Granting CAN_QUERY on {len(target_eps)} owned endpoints")
for ep in target_eps:
    ok = _grant_can_query(ep.id, ep.name, app_sp_id)
    if not ok:
        ep_failures.append(ep.name)

# ── Grant CAN_RUN on owned Genie spaces ───────────────────────────────────────
genie_failures = []
genie_orphans  = []
print(f"Granting CAN_RUN on {len(owned_space_ids)} owned Genie spaces")
for space_id in owned_space_ids:
    result = _grant_can_run_genie(space_id, app_sp_id)
    if result == "failed":
        genie_failures.append(space_id)
    elif result == "orphan":
        genie_orphans.append(space_id)
if genie_orphans:
    print(
        f"\u26a0\ufe0f  Skipped {len(genie_orphans)} orphan Genie space(s) from uc_state "
        f"(deleted or owned by another identity): {genie_orphans}"
    )

# ── Fail loudly if any grants didn't land ─────────────────────────────────────
# Only real failures count — orphans are treated as warnings (stale uc_state).
if ep_failures or genie_failures:
    raise RuntimeError(
        f"Permission grants failed!\n"
        f"  Endpoints: {ep_failures}\n"
        f"  Genie spaces: {genie_failures}\n"
        "Fix the errors above and re-run this cell."
    )
print("\n\u2705 All endpoint and Genie space permissions granted successfully")

In [ ]:
# Grant DATABRICKS_SUPERUSER to the app service principal in the Lakebase Autoscale project.
# This allows the app to connect and create tables on first startup.
try:
    from databricks.sdk.service.postgres import Role, RoleRoleSpec, RoleMembershipRole, RoleIdentityType
    from databricks.sdk.common.types.fieldmask import FieldMask

    production_branch = f"projects/{OPS_PROJECT_ID}/branches/production"
    app_principal = app_sp_id  # resolved service principal client ID

    # Find or create a role for the app SP
    existing_roles = list(w.postgres.list_roles(production_branch))
    app_role = next(
        (r for r in existing_roles if getattr(r.spec, "postgres_role", None) == app_principal),
        None,
    )

    if app_role:
        app_role.spec = RoleRoleSpec(
            identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
            postgres_role=app_principal,
            membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
        )
        w.postgres.update_role(
            name=app_role.name,
            role=app_role,
            update_mask=FieldMask(field_mask=["spec.membership_roles"]),
        )
        print(f"\u2705 Updated app role to DATABRICKS_SUPERUSER")
    else:
        new_role = w.postgres.create_role(
            parent=production_branch,
            role=Role(
                spec=RoleRoleSpec(
                    identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                    postgres_role=app_principal,
                    membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
                ),
            ),
        )
        print(f"\u2705 Created DATABRICKS_SUPERUSER role for app SP: {app_principal}")
except Exception as e:
    print(f"\u26a0\ufe0f Could not grant Lakebase role to app SP: {e}")

##### Write app.yaml — assembles all resource IDs from uc_state before deploy

In [ ]:
import json as _json, re as _re, os as _os

def _latest_from_uc_state(resource_type, key):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = '{resource_type}'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            val = _json.loads(row.resource_data).get(key, "")
            if val:
                return val
    except Exception as e:
        print(f"⚠️ uc_state lookup failed for {resource_type}/{key}: {e}")
    return ""

def _ka_tile_id(ka_name):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = 'knowledge_assistants'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = _json.loads(row.resource_data)
            if info.get("name") == ka_name:
                return info.get("tile_id", "")
    except Exception as e:
        print(f"⚠️ Could not read KA tile {ka_name}: {e}")
    return ""

# Supervisor
supervisor_endpoint = _latest_from_uc_state("multi_agent_supervisors", "endpoint_name")
supervisor_tile_id  = _latest_from_uc_state("multi_agent_supervisors", "tile_id")

supervisor_mlflow_experiment_id = ""
if supervisor_tile_id:
    try:
        _tile = w.api_client.do("GET", f"/api/2.0/tiles/{supervisor_tile_id}")
        supervisor_mlflow_experiment_id = str(_tile.get("mlflow_experiment_id") or "")
    except Exception as e:
        print(f"⚠️ Could not fetch mlflow_experiment_id from tile: {e}")

# Genie spaces (titles must match stages/genie_spaces.ipynb)
_revenue_title = f"Revenue & Orders Intelligence ({CATALOG})"
_ops_title     = f"Operations Intelligence ({CATALOG})"
_menu_title    = f"Menu & Safety Intelligence ({CATALOG})"
genie_revenue_id = genie_ops_id = genie_menu_id = ""
try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'genie_spaces'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = _json.loads(row.resource_data)
        title = info.get("title")
        if title == _revenue_title and not genie_revenue_id:
            genie_revenue_id = info.get("space_id", "")
        elif title == _ops_title and not genie_ops_id:
            genie_ops_id = info.get("space_id", "")
        elif title == _menu_title and not genie_menu_id:
            genie_menu_id = info.get("space_id", "")
except Exception as e:
    print(f"⚠️ Could not read genie IDs: {e}")

# KA tile IDs (one per Knowledge Assistant created by stages/knowledge_agents.ipynb)
ka_inspection  = _ka_tile_id(f"{CATALOG}-inspection-knowledge")
ka_menu        = _ka_tile_id(f"{CATALOG}-menu-knowledge")
ka_legal       = _ka_tile_id(f"{CATALOG}-legal")
ka_regulatory  = _ka_tile_id(f"{CATALOG}-regulatory")
ka_audits      = _ka_tile_id(f"{CATALOG}-audits")
ka_consultancy = _ka_tile_id(f"{CATALOG}-consultancy")

# Lakebase endpoint path — deterministic from CATALOG, points at the
# shared Autoscaling project (`<CATALOG>-caspers`) created by
# stages/lakebase_project.ipynb.  Reuses OPS_PROJECT_ID computed at the top
# of this notebook.
lakebase_endpoint_path = OPS_ENDPOINT_PATH

app_yaml_path = _os.path.abspath("../apps/caspers-ops-dashboard/app.yaml")
app_yaml_contents = f"""command:
  - uvicorn
  - app.main:app
env:
  - name: LAKEBASE_ENDPOINT_PATH
    value: '{lakebase_endpoint_path}'
  - name: LAKEBASE_DATABASE_NAME
    value: '{OPS_DATABASE_NAME}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: SUPERVISOR_ENDPOINT
    value: '{supervisor_endpoint}'
  - name: SUPERVISOR_TILE_ID
    value: '{supervisor_tile_id}'
  - name: SUPERVISOR_MLFLOW_EXPERIMENT_ID
    value: '{supervisor_mlflow_experiment_id}'
  - name: GENIE_ID_REVENUE
    value: '{genie_revenue_id}'
  - name: GENIE_ID_OPS
    value: '{genie_ops_id}'
  - name: GENIE_ID_MENU
    value: '{genie_menu_id}'
  - name: KA_ID_INSPECTION
    value: '{ka_inspection}'
  - name: KA_ID_MENU
    value: '{ka_menu}'
  - name: KA_ID_LEGAL
    value: '{ka_legal}'
  - name: KA_ID_REGULATORY
    value: '{ka_regulatory}'
  - name: KA_ID_AUDITS
    value: '{ka_audits}'
  - name: KA_ID_CONSULTANCY
    value: '{ka_consultancy}'
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse.id}'
"""
with open(app_yaml_path, "w") as _f:
    _f.write(app_yaml_contents)

print(f"✅ Wrote app.yaml to {app_yaml_path}")
print(f"   Supervisor endpoint:  {supervisor_endpoint}")
print(f"   Supervisor tile:      {supervisor_tile_id}")
print(f"   MLflow experiment:    {supervisor_mlflow_experiment_id}")
print(f"   Lakebase endpoint:    {lakebase_endpoint_path}")
print(f"   Revenue Genie:        {genie_revenue_id}")
print(f"   Ops Genie:            {genie_ops_id}")
print(f"   Menu Genie:           {genie_menu_id}")
print(f"   KA Inspection:        {ka_inspection}")
print(f"   KA Menu:              {ka_menu}")
print(f"   KA Legal:             {ka_legal}")
print(f"   KA Regulatory:        {ka_regulatory}")
print(f"   KA Audits:            {ka_audits}")
print(f"   KA Consultancy:       {ka_consultancy}")
print(f"   Warehouse:            {warehouse.id}")

In [ ]:
import time

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(source_code_path=source_code_path),
)

def _deploy_state(d):
    st = getattr(d, "status", None)
    s = getattr(st, "state", None) if st is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current_dep = w.apps.get_deployment(app_name=app_status.name, deployment_id=deployment.deployment_id)
    state = _deploy_state(current_dep)
    print(f"Deployment state: {state}")
    if state == "SUCCEEDED":
        deployment_status = current_dep
        break
    if state in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Deployment failed for {app_status.name}: state={state}")
    if time.time() > deadline:
        raise TimeoutError(f"Deployment for {app_status.name} not ready after 30 minutes (last state: {state})")
    time.sleep(10)

print(f"\u2705 Operational Dashboard deployed")
print(f"   URL: {app_status.url if hasattr(app_status, 'url') else 'Check Databricks Apps UI'}")
display(deployment_status)